# Mass-Radius Relationships

This notebook explores how planet radius depends on mass and composition.

We'll generate mass-radius curves for different planet types:
- Rocky planets (iron core + silicate mantle)
- Water-rich planets (rocky core + water layer)
- Gas-rich planets (with atmosphere)

In [ ]:
import magrathea as mag
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## Mass-Radius Curve for Rocky Planets

First, let's create a mass-radius relationship for pure rocky planets (33% iron core, 67% silicate mantle).

In [ ]:
def compute_rocky_planet_curve(masses, core_fraction=0.33):
    """
    Compute mass-radius curve for rocky planets
    
    Parameters:
    - masses: array of total masses (Earth masses)
    - core_fraction: fraction of mass in iron core
    
    Returns:
    - radii: array of planet radii (Earth radii)
    """
    radii = []
    
    for mass in masses:
        # Create planet
        planet = mag.Planet()
        planet.add_layer('core', mass=mass * core_fraction)
        planet.add_layer('mantle', mass=mass * (1 - core_fraction))
        planet.surface_temp = 300
        
        try:
            planet.solve()
            radii.append(planet.results.radius)
            print(f"M = {mass:.2f} M⊕ → R = {planet.results.radius:.4f} R⊕")
        except Exception as e:
            print(f"M = {mass:.2f} M⊕ → Failed: {str(e)[:50]}")
            radii.append(np.nan)
    
    return np.array(radii)

# Define mass range
masses_rocky = np.array([0.5, 1.0, 2.0, 5.0, 10.0])

print("Computing rocky planet mass-radius curve...")
print("This may take a minute or two...\n")

try:
    radii_rocky = compute_rocky_planet_curve(masses_rocky)
except Exception as e:
    print(f"Error computing curve: {e}")
    radii_rocky = np.zeros_like(masses_rocky) * np.nan

## Plot Mass-Radius Curve

In [ ]:
if not np.all(np.isnan(radii_rocky)):
    fig, ax = plt.subplots(figsize=(10, 7))
    
    # Plot mass-radius curve
    valid_mask = ~np.isnan(radii_rocky)
    ax.plot(masses_rocky[valid_mask], radii_rocky[valid_mask], 
            'o-', linewidth=2.5, markersize=10, label='Rocky Planets (33% Fe core)')
    
    # Mark Earth
    ax.plot([1.0], [1.0], 'g*', markersize=20, label='Earth', zorder=10)
    
    ax.set_xlabel('Planet Mass (M⊕)', fontsize=14)
    ax.set_ylabel('Planet Radius (R⊕)', fontsize=14)
    ax.set_title('Mass-Radius Relationship for Rocky Planets', 
                 fontsize=16, fontweight='bold')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3, which='both')
    ax.legend(fontsize=12)
    
    plt.tight_layout()
    plt.show()
else:
    print("Could not generate plot. Ensure Magrathea C++ executable is built.")

## Compare Different Core Fractions

Let's see how the core mass fraction affects the mass-radius relationship.

In [ ]:
# Smaller mass range for faster computation
masses_comp = np.array([0.5, 1.0, 2.0])
core_fractions = [0.1, 0.33, 0.5, 0.7]

fig, ax = plt.subplots(figsize=(10, 7))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(core_fractions)))

for i, core_frac in enumerate(core_fractions):
    print(f"\nCore fraction: {core_frac}")
    try:
        radii = compute_rocky_planet_curve(masses_comp, core_fraction=core_frac)
        
        valid_mask = ~np.isnan(radii)
        if np.any(valid_mask):
            ax.plot(masses_comp[valid_mask], radii[valid_mask], 
                   'o-', linewidth=2, markersize=8, 
                   color=colors[i], label=f'Core fraction = {core_frac}')
    except:
        print(f"Failed for core fraction {core_frac}")

# Mark Earth
ax.plot([1.0], [1.0], 'k*', markersize=20, label='Earth', zorder=10)

ax.set_xlabel('Planet Mass (M⊕)', fontsize=14)
ax.set_ylabel('Planet Radius (R⊕)', fontsize=14)
ax.set_title('Effect of Core Composition on Mass-Radius Relation', 
             fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='best')

plt.tight_layout()
plt.show()

print("\nKey insight: Denser cores (higher iron fraction) lead to smaller radii for the same mass.")

## Water-Rich Planets

Now let's add a water layer to see how it affects planet radius.

In [ ]:
def compute_water_planet_radius(total_mass, water_fraction):
    """
    Compute radius of a planet with a water layer
    
    Parameters:
    - total_mass: total planet mass (Earth masses)
    - water_fraction: fraction of mass in water
    """
    planet = mag.Planet()
    
    # Rocky interior (70% of rocky part is mantle, 30% is core)
    rocky_mass = total_mass * (1 - water_fraction)
    planet.add_layer('core', mass=rocky_mass * 0.33)
    planet.add_layer('mantle', mass=rocky_mass * 0.67)
    planet.add_layer('hydro', mass=total_mass * water_fraction)
    
    planet.surface_temp = 300
    
    try:
        planet.solve()
        return planet.results.radius
    except:
        return np.nan

# Compare 1 Earth mass planet with different water fractions
water_fractions = [0.0, 0.1, 0.25, 0.5]
radii_water = []

print("Computing water-rich planet radii...\n")
for wf in water_fractions:
    r = compute_water_planet_radius(1.0, wf)
    radii_water.append(r)
    print(f"Water fraction {wf:.2f} → Radius = {r:.4f} R⊕")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

valid_mask = ~np.isnan(radii_water)
if np.any(valid_mask):
    ax.plot(np.array(water_fractions)[valid_mask], 
            np.array(radii_water)[valid_mask], 
            'bo-', linewidth=2.5, markersize=10)
    
    ax.set_xlabel('Water Mass Fraction', fontsize=14)
    ax.set_ylabel('Planet Radius (R⊕)', fontsize=14)
    ax.set_title('Effect of Water Content on Planet Radius\n(1 Earth Mass)', 
                 fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nKey insight: Water layers significantly increase planet radius due to lower density.")
else:
    print("Could not generate water planet comparison.")

## Summary

In this notebook, we explored:

1. **Mass-radius relationships** for rocky planets
2. How **core composition** affects planet size
3. The dramatic effect of **water layers** on planet radius

These relationships are crucial for:
- Understanding exoplanet observations
- Inferring planetary composition from mass and radius measurements
- Studying planet formation and evolution